In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
import os
import typer
import json
from loguru import logger
from tqdm import tqdm
import datetime
import functools
from pathlib import Path
import concurrent.futures

import numpy as np
import torch
import torch.nn.functional

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import PreTrainedTokenizerFast, DataCollatorForLanguageModeling
from transformers.data import data_collator

from translast.loaders.tokenizer import GenomeIterator, train_sentencepiece, _kmer_split, _dataset_batch
from translast.modeling.albert import AlbertConfig, AlbertModel, AlbertMaskedWrapper, AlbertMaskedTrainer
from translast.modeling.train import TrainerConfig, Trainer
from translast.config import MODELS_DIR, PROCESSED_DATA_DIR


In [3]:
transcripts = '../data/raw/gencode.v47.transcripts.fa.gz'
raw_transcripts = GenomeIterator(transcripts, 'fasta')

In [4]:
logger.info("Training ALBERT model...")
trainer_config = TrainerConfig.from_json("../translast/modeling/trainer_config.json")
albert_config = AlbertConfig.from_json("../translast/modeling/albert_config.json")
model = AlbertModel(albert_config)
data_iter = ...  # Replace with your data iterator
# save_dir = "models"
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# trainer = Trainer(trainer_config, model, data_iter, save_dir, device)
# trainer.train(loss_function, data_parallel=True)
# for i in tqdm(range(10), total=10):
#     if i == 5:
#         logger.info("Something happened for iteration 5.")
logger.success("Modeling training complete.")
# # -----------------------------------------

2025-03-11 16:47:29.838 | INFO     | __main__:<module>:1 - Training ALBERT model...
2025-03-11 16:47:29.877 | SUCCESS  | __main__:<module>:13 - Modeling training complete.


In [5]:
log = lambda s: print(f"> {s}") # TODO: Colour!

"""
    Create workspace and return pointers 
    to the created directories
"""
def create_workspace(save_id):
    runs_dir = Path("runs")
    root_dir = runs_dir / f"pretrain-{save_id}"
    chk_dir = root_dir / "checkpoints"
    log_dir = root_dir / "log_dir"

    runs_dir.mkdir(exist_ok=True)
    root_dir.mkdir(exist_ok=True)
    chk_dir.mkdir(exist_ok=True)
    log_dir.mkdir(exist_ok=True)
    
    return root_dir, chk_dir, log_dir

"""
    Loading English Wikipedia dataset
    1. Download dataset
    2. Remove title column
    3. Chunk text into smaller sequences
    4. Filter shorter sequences
    5. Create train-test split
"""
def filter_dataset(generator, filter_function, num_workers):
    # Use ThreadPoolExecutor for parallel processing
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
        # Apply the filter function to the generator
        filtered_data = list(executor.map(lambda e: e if filter_function(e) else None, generator))
    
    # Remove None values from the filtered data
    filtered_data = [e for e in filtered_data if e is not None]
    
    return filtered_data
    
def dataset_loader(builder = '../data/raw/gencode.v47.transcripts.fa.gz', k=17, batch_size=1000, chunk_size=2000, test_split=.1, num_workers = 4):
    log(f"Loading transcripts {builder}.")
    raw_datasets = GenomeIterator(builder, 'fasta')

    def generator_from_iterator(raw_datasets):
        for seq in raw_datasets:
            yield {'sequence': seq, 'kmers': _kmer_split(k, seq)}

    def _chunk_text(batch, chunk_size=2000):
        chunks = []
        for s in batch['kmers']:
            chunks += [s[i:i+chunk_size] for i in range(0, len(s), chunk_size)]
        return {'chunks': chunks}

    log("Create datasets from GenomeIterator")
    dataset = Dataset.from_generator(generator_from_iterator, gen_kwargs={"raw_datasets": raw_datasets})
    
    log("Chunking text to maximum sequence_length")
    dataset = dataset.map(functools.partial(_chunk_text, chunk_size=chunk_size), batched=True, num_proc=num_workers, remove_columns=dataset.column_names)
    
    log("Filtering short sequences")
    dataset = dataset.filter(lambda e: len(e['chunks']) >= chunk_size, num_proc=num_workers)

    log("Creating train-eval split")
    dataset = dataset.train_test_split(test_size=test_split)

    return dataset

"""
    Given a batch, process it into a batch of tensors
    1. Pick a random split
    2. Pick a random ordering
    3. Tokenize sentence pairs
    4. Mask tokens
    5. Cast to tensors

    TODO: Generally, make this more efficient
"""
def process_batch(batch, tokenizer, collator, mask_prob, chunk_size=2000, max_length=1024):
    chunks = batch['chunks']
    batch_size = len(chunks)
    random_deltas = torch.randint(-chunk_size // 4, chunk_size // 4, (batch_size,))

    mid = chunk_size // 2

    sentence_pairs = [(c[:mid+random_deltas[i]], c[mid+random_deltas[i]:]) for i, c in enumerate(chunks)]
    ordering = torch.randint(0, 2, (batch_size,))

    sentence_pairs = [(c[0],c[1]) if ordering[i] else (c[1], c[0]) for i, c in enumerate(sentence_pairs)]

    tokenized_pairs = tokenizer([s0 for (s0,_) in sentence_pairs], [s1 for (_,s1) in sentence_pairs], 
                        padding='max_length', max_length=max_length, pad_to_multiple_of=8, truncation=True,
                        return_special_tokens_mask=True, return_tensors='pt')
    
    masked_input, token_labels = collator.torch_mask_tokens(tokenized_pairs['input_ids'], tokenized_pairs['special_tokens_mask'])

    return masked_input,\
           tokenized_pairs['token_type_ids'],\
           tokenized_pairs['attention_mask'].bool(),\
           token_labels,\
           ordering

class EpochMetric:
    def __init__(self):
        self.loss = 0.0

        self.cls_loss = 0.0
        self.cls_accuracy = 0.0

        self.token_loss = 0.0
        self.token_accuracy = 0.0

        self.nb_updates = 0

    def update(self, loss, cls, token):
        self.loss += loss

        self.cls_loss += cls[0]
        self.cls_accuracy += cls[1]

        self.token_loss += token[0]
        self.token_accuracy += token[1]

        self.nb_updates += 1

    def __str__(self):
        s = ""
        s += f"loss: {self.loss / self.nb_updates} "
        s += f"| cls: [loss: {self.cls_loss / self.nb_updates}, accuracy: {100.0 * self.cls_accuracy / self.nb_updates:.2f}%] "
        s += f"| token: [loss: {self.token_loss / self.nb_updates}, accuracy: {100.0 * self.token_accuracy / self.nb_updates:.2f}%]"
        return s

In [6]:
device = torch.device(device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
log(f"device: {device.type}")

save_id = str(datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))

logger.info("Loading config ALBERT transLAST model...")

trainer_config = TrainerConfig.from_json("../translast/modeling/trainer_config.json")
albert_config = AlbertConfig.from_json("../translast/modeling/albert_config.json")

log("Loading pretrained tokenizer")
tokenizer = PreTrainedTokenizerFast.from_pretrained(pretrained_model_name_or_path = "transcripts_sentencepiece/unigram/huggingface/gencode.v47.transcripts.k17",
                                                    local_files_only=True)
albert_config.vocab_size = len(tokenizer) + (8 - len(tokenizer) % 8) # pad to multiple of 8 for tensor core optimisation
trainer_config.mask_prob = 0.15

log("Preparing DataCollatorForLanguageModeling")
collator = DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=trainer_config.mask_prob)


> device: mps
2025-03-11 16:47:29.931 | INFO     | __main__:<module>:6 - Loading config ALBERT transLAST model...
> Loading pretrained tokenizer
> Preparing DataCollatorForLanguageModeling


In [7]:
trainer_config.chunk_size = 2000
trainer_config.test_size = 0.1
trainer_config.small = True

# TODO: Convert to dataloader version
log("Load and preprocessing dataset")
dataset = dataset_loader(builder = '../data/raw/gencode.v47.transcripts.fa.gz', k=17, 
                       chunk_size=trainer_config.chunk_size, test_split=trainer_config.test_size)

> Load and preprocessing dataset
> Loading transcripts ../data/raw/gencode.v47.transcripts.fa.gz.
> Create datasets from GenomeIterator
> Chunking text to maximum sequence_length
> Filtering short sequences
> Creating train-eval split


In [8]:
idx = np.random.choice(len(dataset['train']), trainer_config.batch_size)
b = dataset['train'][idx[1*trainer_config.mini_batch_size:(2)*trainer_config.mini_batch_size]]

t = process_batch(b, tokenizer, collator, trainer_config.mask_prob, max_length=albert_config.max_position_embeddings)

In [9]:
trainer = AlbertMaskedTrainer(albert_config, trainer_config)
metrics = trainer.train_step(t)
metrics

(tensor(19.4947, grad_fn=<AddBackward0>),
 (tensor(0.6677, grad_fn=<NllLossBackward0>), tensor(0.5938)),
 (tensor(18.8270, grad_fn=<NllLossBackward0>), tensor(0.)))

In [10]:
def train_epoch(trainer, dataset, tokenizer, collator, config, args):
    epoch_metrics = EpochMetric()
    update_frequency = config.batch_size // config.mini_batch_size
    pb = tqdm(range(32), disable= not args.tqdm)
    
    for _ in pb:
        idx = np.random.choice(len(dataset['train']), config.batch_size)
        
        for i in range(update_frequency):
            b = dataset['train'][idx[i*config.mini_batch_size:(i+1)*config.mini_batch_size]]
            b = process_batch(b, tokenizer, collator, config.mask_prob, max_length=trainer.config.max_position_embeddings)
            metrics = trainer.train_step(b)
            epoch_metrics.update(*metrics)
        
        if not args.no_save and (trainer.ts + 1) % config.save_frequency == 0:
            trainer.save_checkpoint(args.chk_dir / f"albert-{args.model}-checkpoint-{str(trainer.ts).zfill(7)}.pt")
        
        trainer.ts += 1
        display = f"training | ts: {str(trainer.ts).zfill(7)} | {str(epoch_metrics)}"
        pb.set_description(display)
        pb.update(1)
    
    if not args.tqdm:
        log(display)
    
    return epoch_metrics

def evaluate_epoch(trainer, dataset, tokenizer, collator, config, args):
    epoch_metrics = EpochMetric()
    pb = tqdm(range(4 * (config.batch_size // config.mini_batch_size)), disable= not args.tqdm)
    
    for _ in pb:
        idx = np.random.choice(len(dataset['test']), config.mini_batch_size)
        b = dataset['test'][idx]
        b = process_batch(b, tokenizer, collator, config.mask_prob, max_length=trainer.config.max_position_embeddings)
        metrics = trainer.eval_step(b)
        epoch_metrics.update(*metrics)
        
        display = f"evaluation | ts: {str(trainer.ts).zfill(7)} | {str(epoch_metrics)}"
        pb.set_description(display)
        pb.update(1)
    
    if not args.tqdm:
        log(display)
    
    return epoch_metrics

In [11]:
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from argparse import Namespace

args = Namespace(tqdm=True, model='albertLAST', no_save=False)

if not args.no_save:
    args.root_dir, args.chk_dir, args.log_dir = create_workspace(save_id)

trainer = AlbertMaskedTrainer(albert_config, trainer_config)
trainer.ts = 0

In [13]:
while trainer.ts < 125000:
    train_epoch(trainer, dataset, tokenizer, collator, trainer_config, args)
    evaluate_epoch(trainer, dataset, tokenizer, collator, trainer_config, args)

training | ts: 0000031 | loss: 11.457056045532227 | cls: [loss: 0.7265006303787231, accuracy: 49.62%] | token: [loss: 10.730555534362793, accuracy: 2.11%]:  97%|█████████▋| 31/32 [10:31<00:20, 20.37s/it]


KeyboardInterrupt: 

In [35]:
import unicodedata